<div style="text-align: center; padding: 80px 40px; border: 3px solid #1a1a2e; border-radius: 8px; background: linear-gradient(135deg, #f8f9fa 0%, #e8eef5 100%);">

<h1 style="font-size: 2.4em; color: #1a1a2e; margin-bottom: 10px; letter-spacing: 1px;">Time Series Analysis Report</h1>

<hr style="width: 40%; border: 1px solid #4a6fa5; margin: 30px auto;">

<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>Students:</strong> Ulysse LONG &amp; Camille PECH</p>
<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>Class:</strong> FGE5</p>
<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>School:</strong> EPF</p>

<hr style="width: 40%; border: 1px solid #4a6fa5; margin: 30px auto;">

<p style="font-size: 1.1em; color: #555; font-style: italic; margin-top: 20px;">
Google Trends time series analysis<br>
Search query « Idée de cadeau » — France (2004 – 2026)
</p>

</div>

## Table of Contents

1. [Definition of Interest (Google Trends)](#definition-of-interest-google-trends)
2. [Setting up the environment](#setting-up-the-environment)
3. [Dataset visualization](#dataset-visualization)
4. [Decomposition model — Buys-Ballot test](#decomposition-model--buys-ballot-test)
5. [Trend detection — Fisher row-effects test](#trend-detection--fisher-row-effects-test)
6. [Seasonality detection — Fisher column-effects test](#seasonality-detection--fisher-column-effects-test)
7. [Decomposition order — variance impact](#decomposition-order--variance-impact)

## Definition of Interest (Google Trends)

**Interest** for a search query over a given period is shown on a **relative scale from 0 to 100**, where:

- **100** corresponds to the peak interest for the time range of the chart;
- **50** means the term is half as popular as the peak;
- **0** means there is not enough data for that period.

In this report, we analyze the evolution of interest for the Google query **« Idée de cadeau »** in France, from January 2004 to October 2026.

## Setting up the environment

The cell below creates a local Python virtual environment (`.venv`), installs all required libraries, registers a Jupyter kernel for this project, and generates a `requirements.txt` file for GitHub.

**Run it once**, then select the kernel **Python (TSA-EPF)** via *Kernel ▸ Change Kernel*.  
Do **not** push the `.venv` folder to GitHub — only `requirements.txt`.

In [ ]:
# Virtual environment setup (.venv) — run once
# Do NOT push .venv to GitHub (see .gitignore)

import subprocess
import sys
from pathlib import Path

VENV_DIR = Path(".venv")
PYTHON = sys.executable
PACKAGES = [
    "pandas",
    "matplotlib",
    "numpy",
    "statsmodels",
    "scikit-learn",
    "plotly",
    "ipykernel",
]

# 1. Create the venv if it does not exist
if not VENV_DIR.exists():
    print("Creating virtual environment (.venv)...")
    subprocess.check_call([PYTHON, "-m", "venv", str(VENV_DIR)])
else:
    print(".venv already exists — reusing it.")

# Paths to the venv Python / pip (macOS / Linux / Windows)
if sys.platform == "win32":
    venv_python = VENV_DIR / "Scripts" / "python.exe"
else:
    venv_python = VENV_DIR / "bin" / "python"

# 2. Install required packages into the venv
print("Installing packages...")
subprocess.check_call(
    [str(venv_python), "-m", "pip", "install", "--upgrade", "pip"]
)
subprocess.check_call(
    [str(venv_python), "-m", "pip", "install", *PACKAGES]
)

# 3. Register a Jupyter kernel for this project
subprocess.check_call(
    [
        str(venv_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "tsa-epf",
        "--display-name",
        "Python (TSA-EPF)",
    ]
)

# 4. Write a lean requirements.txt (direct deps only — not a full pip freeze)
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(PACKAGES) + "\n")

print("\n✅ Environment ready.")
print("→ Select the kernel: Kernel ▸ Change Kernel ▸ Python (TSA-EPF)")
print("→ On GitHub: push requirements.txt, ignore .venv/")

## Dataset visualization

The cell below loads the Google Trends CSV and displays an **interactive** chart of monthly interest for the query **« Idée de cadeau »** in France.

You can **zoom** (scroll or box-select), **pan**, and use the **range slider** under the chart to focus on a specific period. Double-click to reset the view.

In [3]:
import pandas as pd
import plotly.graph_objects as go

# Load the dataset
df = pd.read_csv(
    "time_series_FR_20040101-0100_20261001-1602.csv",
    parse_dates=["Time"],
)
df = df.rename(columns={"Time": "Date", "Idée de cadeau": "Interest"})
df = df.set_index("Date")

# Interactive plot (zoom, pan, period selection)
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Interest"],
        mode="lines",
        name="Interest",
        line=dict(color="#2c5aa0", width=2),
        fill="tozeroy",
        fillcolor="rgba(44, 90, 160, 0.15)",
        hovertemplate="Date: %{x|%Y-%m}<br>Interest: %{y}<extra></extra>",
    )
)

fig.update_layout(
    title="Google Trends Interest — « Idée de cadeau » (France, 2004–2026)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    yaxis=dict(title="Interest (scale 0–100)", range=[0, 105]),
    xaxis=dict(
        title="Date",
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=[
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=5, label="5y", step="year", stepmode="backward"),
                dict(count=10, label="10y", step="year", stepmode="backward"),
                dict(step="all", label="All"),
            ]
        ),
    ),
)

fig.show()

print(f"Period: {df.index.min().strftime('%Y-%m')} → {df.index.max().strftime('%Y-%m')}")
print(f"Number of observations: {len(df)}")
print(f"Average interest: {df['Interest'].mean():.1f}")
print(f"Max interest: {df['Interest'].max()} ({df['Interest'].idxmax().strftime('%Y-%m')})")

Period: 2004-01 → 2026-09
Number of observations: 273
Average interest: 23.9
Max interest: 100 (2013-12)


## Decomposition model — Buys-Ballot test

Buys-Ballot table: each **row** = one period (year), each **column** = one season (month).

The Buys-Ballot test evaluates the linear relationship between the **periodic average** and the **periodic standard deviation**:

$$\sigma_{p_i} = \alpha + \beta\,\bar{p}_i + \varepsilon_i$$

- If $\beta$ is statistically null → **additive** model
- If $\beta$ is significantly non-null → **multiplicative** model

In [4]:
import numpy as np
import statsmodels.api as sm

# --- Buys-Ballot table (years × months) ---
bb = df.copy()
bb["year"] = bb.index.year
bb["month"] = bb.index.month

# Keep complete periods only (12 months), as required for a full BB table
complete_years = bb.groupby("year").size()
complete_years = complete_years[complete_years == 12].index
bb = bb[bb["year"].isin(complete_years)]

table = bb.pivot_table(index="year", columns="month", values="Interest")

# Period (row) averages and standard deviations
period_avg = table.mean(axis=1)   # p̄_i
period_std = table.std(axis=1)    # σ_p_i

# Season (column) averages and standard deviations
season_avg = table.mean(axis=0)   # s̄_j
season_std = table.std(axis=0)    # σ_s_j

# Display BB table with period / seasonal averages and stds
bb_display = table.copy()
bb_display["Average (p̄_i)"] = period_avg
bb_display["StdDev (σ_p_i)"] = period_std
bb_display.loc["Average (s̄_j)"] = list(season_avg) + [np.nan, np.nan]
bb_display.loc["StdDev (σ_s_j)"] = list(season_std) + [np.nan, np.nan]

print("Buys-Ballot table")
display(bb_display.round(2))

# --- Buys-Ballot test: σ_p_i = α + β p̄_i + ε_i ---
X = sm.add_constant(period_avg)
ols = sm.OLS(period_std, X).fit()

beta = ols.params.iloc[1]
p_value = ols.pvalues.iloc[1]

print(ols.summary())
print(f"\nβ = {beta:.4f}")
print(f"p-value (β) = {p_value:.4f}")

# Decision rule from the lecture:
# - β statistically null      → additive
# - β significantly non-null → multiplicative
if p_value >= 0.05:
    decomposition_model = "additive"
    print("\n→ β is statistically null → additive model (X_t = T_t + S_t + R_t)")
else:
    decomposition_model = "multiplicative"
    print("\n→ β is significantly non-null → multiplicative model (X_t = T_t × S_t × R_t)")

print(f"\nSelected decomposition model: {decomposition_model}")

Buys-Ballot table


month,1,2,3,4,5,6,7,8,9,10,11,12,Average (p̄_i),StdDev (σ_p_i)
year,,,,,,,,,,,,,,
2004,0.00,7.00,0.00,11.00,6.00,12.00,0.00,6.00,8.00,8.00,15.00,37.00,9.17,10.00
2005,5.00,9.00,4.00,8.00,11.00,9.00,4.00,7.00,7.00,8.00,20.00,38.00,10.83,9.54
2006,6.00,11.00,6.00,5.00,11.00,8.00,6.00,8.00,5.00,9.00,21.00,43.00,11.58,10.82
2007,8.00,11.00,7.00,6.00,11.00,12.00,9.00,9.00,10.00,12.00,27.00,46.00,14.00,11.42
2008,7.00,13.00,8.00,8.00,15.00,12.00,9.00,10.00,12.00,16.00,39.00,57.00,17.17,15.16
2009,8.00,13.00,10.00,10.00,14.00,17.00,11.00,12.00,12.00,16.00,36.00,61.00,18.33,15.28
2010,9.00,16.00,10.00,12.00,20.00,17.00,11.00,14.00,14.00,16.00,40.00,70.00,20.75,17.51
2011,10.00,16.00,11.00,14.00,23.00,20.00,16.00,18.00,17.00,21.00,51.00,87.00,25.33,22.12
2012,16.00,22.00,17.00,14.00,21.00,23.00,15.00,16.00,18.00,22.00,48.00,74.00,25.50,17.74


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.821
Model:                            OLS   Adj. R-squared:                  0.812
Method:                 Least Squares   F-statistic:                     91.60
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           6.58e-09
Time:                        17:04:11   Log-Likelihood:                -46.421
No. Observations:                  22   AIC:                             96.84
Df Residuals:                      20   BIC:                             99.02
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.8303      1.594      2.402      0.0

## Trend detection — Fisher row-effects test

Periodic influence (row effect) test on the Buys-Ballot table:

$$H_0:\ \text{no influence}$$

$$F_c = \frac{V_p}{V_R}$$

Compare $F_c$ to a Fisher critical value $\mathcal{F}_{\nu_1;\nu_2}$ with $\nu_1 = P - 1$ and $\nu_2 = (P - 1)(S - 1)$.

If $F_c > \mathcal{F}_{\nu_1;\nu_2}$, then $H_0$ is rejected → **presence of a trend component**.

In [6]:
from scipy import stats

# Buys-Ballot matrix (complete years × months) — same setup as the lecture ANOVA
bb_tt = df.copy()
bb_tt["year"] = bb_tt.index.year
bb_tt["month"] = bb_tt.index.month

counts = bb_tt.groupby("year").size()
complete = counts[counts == 12].index
bb_tt = bb_tt[bb_tt["year"].isin(complete)]

table_tt = bb_tt.pivot_table(index="year", columns="month", values="Interest")
X = table_tt.values  # shape (P, S)

P, S = X.shape  # P = number of periods (years), S = number of seasons (months)

# Averages from the lecture
X_p = X.mean(axis=1)   # X_{p,∘}  periodic (row) averages
X_s = X.mean(axis=0)   # X_{∘,s}  seasonal (column) averages
X_oo = X.mean()        # X_{∘∘}   grand mean

# Variance analysis (row / periodic effect)
# S_p = S * Σ_p (X_{p,∘} - X_{∘∘})² ,  df = P - 1 ,  V_p = S_p / (P - 1)
S_p = S * np.sum((X_p - X_oo) ** 2)
df_p = P - 1
V_p = S_p / df_p

# S_R = Σ_p Σ_s (X_{p,s} - X_{p,∘} - X_{∘,s} + X_{∘∘})² ,  df = (P-1)(S-1)
S_R = np.sum((X - X_p[:, None] - X_s[None, :] + X_oo) ** 2)
df_R = (P - 1) * (S - 1)
V_R = S_R / df_R

# Fisher statistic
F_c = V_p / V_R
nu1 = P - 1
nu2 = (P - 1) * (S - 1)
F_crit = stats.f.ppf(1 - 0.05, nu1, nu2)  # F_{ν1;ν2} at the usual 5% level

print("=" * 60)
print("Periodic influence (row effect) test — Trend detection")
print("=" * 60)
print("\nHypotheses:")
print("  H0 : no periodic influence  →  no trend component")
print("  H1 : periodic influence     →  presence of trend component T_t")
print(f"\nDecision rule: reject H0 if F_c > F_{{ν1;ν2}} (α = 0.05)")
print(f"               i.e. reject H0 if F_c > {F_crit:.4f}")

print("\nResults:")
print(f"  P = {P} periods,  S = {S} seasons")
print(f"  ν1 = P - 1 = {nu1}")
print(f"  ν2 = (P - 1)(S - 1) = {nu2}")
print(f"  V_p = {V_p:.4f}")
print(f"  V_R = {V_R:.4f}")
print(f"  F_c = V_p / V_R = {F_c:.4f}")
print(f"  F critical (ν1={nu1}, ν2={nu2}, α=0.05) = {F_crit:.4f}")

if F_c > F_crit:
    trend_present = True
    print("\nConclusion:")
    print(f"  F_c = {F_c:.4f} > F critical = {F_crit:.4f}")
    print("  → H0 is rejected")
    print("  → Presence of a trend component T_t")
else:
    trend_present = False
    print("\nConclusion:")
    print(f"  F_c = {F_c:.4f} ≤ F critical = {F_crit:.4f}")
    print("  → H0 is not rejected")
    print("  → No significant trend component")

print(f"\nTrend present: {trend_present}")
print("=" * 60)

Periodic influence (row effect) test — Trend detection

Hypotheses:
  H0 : no periodic influence  →  no trend component
  H1 : periodic influence     →  presence of trend component T_t

Decision rule: reject H0 if F_c > F_{ν1;ν2} (α = 0.05)
               i.e. reject H0 if F_c > 1.6018

Results:
  P = 22 periods,  S = 12 seasons
  ν1 = P - 1 = 21
  ν2 = (P - 1)(S - 1) = 231
  V_p = 619.4120
  V_R = 30.0354
  F_c = V_p / V_R = 20.6228
  F critical (ν1=21, ν2=231, α=0.05) = 1.6018

Conclusion:
  F_c = 20.6228 > F critical = 1.6018
  → H0 is rejected
  → Presence of a trend component T_t

Trend present: True


## Seasonality detection — Fisher column-effects test

Seasonal influence (column effect) test on the Buys-Ballot table:

$$H_0:\ \text{no influence}$$

$$F_c = \frac{V_s}{V_R}$$

Compare $F_c$ to a Fisher critical value $\mathcal{F}_{\nu_1;\nu_2}$ with $\nu_1 = S - 1$ and $\nu_2 = (P - 1)(S - 1)$.

If $F_c > \mathcal{F}_{\nu_1;\nu_2}$, then $H_0$ is rejected → **presence of a seasonal component**.

In [7]:
from scipy import stats

# Buys-Ballot matrix (complete years × months)
bb_st = df.copy()
bb_st["year"] = bb_st.index.year
bb_st["month"] = bb_st.index.month

counts = bb_st.groupby("year").size()
complete = counts[counts == 12].index
bb_st = bb_st[bb_st["year"].isin(complete)]

table_st = bb_st.pivot_table(index="year", columns="month", values="Interest")
X = table_st.values  # shape (P, S)

P, S = X.shape

# Averages from the lecture
X_p = X.mean(axis=1)   # X_{p,∘}
X_s = X.mean(axis=0)   # X_{∘,s}
X_oo = X.mean()        # X_{∘∘}

# Variance analysis (column / seasonal effect)
# S_s = P * Σ_s (X_{∘,s} - X_{∘∘})² ,  df = S - 1 ,  V_s = S_s / (S - 1)
S_s = P * np.sum((X_s - X_oo) ** 2)
df_s = S - 1
V_s = S_s / df_s

# S_R = Σ_p Σ_s (X_{p,s} - X_{p,∘} - X_{∘,s} + X_{∘∘})² ,  df = (P-1)(S-1)
S_R = np.sum((X - X_p[:, None] - X_s[None, :] + X_oo) ** 2)
df_R = (P - 1) * (S - 1)
V_R = S_R / df_R

# Fisher statistic
F_c = V_s / V_R
nu1 = S - 1
nu2 = (P - 1) * (S - 1)
F_crit = stats.f.ppf(1 - 0.05, nu1, nu2)

print("=" * 60)
print("Seasonal influence (column effect) test — Seasonality detection")
print("=" * 60)
print("\nHypotheses:")
print("  H0 : no seasonal influence  →  no seasonal component")
print("  H1 : seasonal influence     →  presence of seasonal component S_t")
print(f"\nDecision rule: reject H0 if F_c > F_{{ν1;ν2}} (α = 0.05)")
print(f"               i.e. reject H0 if F_c > {F_crit:.4f}")

print("\nResults:")
print(f"  P = {P} periods,  S = {S} seasons")
print(f"  ν1 = S - 1 = {nu1}")
print(f"  ν2 = (P - 1)(S - 1) = {nu2}")
print(f"  V_s = {V_s:.4f}")
print(f"  V_R = {V_R:.4f}")
print(f"  F_c = V_s / V_R = {F_c:.4f}")
print(f"  F critical (ν1={nu1}, ν2={nu2}, α=0.05) = {F_crit:.4f}")

if F_c > F_crit:
    seasonality_present = True
    print("\nConclusion:")
    print(f"  F_c = {F_c:.4f} > F critical = {F_crit:.4f}")
    print("  → H0 is rejected")
    print("  → Presence of a seasonal component S_t")
else:
    seasonality_present = False
    print("\nConclusion:")
    print(f"  F_c = {F_c:.4f} ≤ F critical = {F_crit:.4f}")
    print("  → H0 is not rejected")
    print("  → No significant seasonal component")

print(f"\nSeasonality present: {seasonality_present}")
print("=" * 60)

Seasonal influence (column effect) test — Seasonality detection

Hypotheses:
  H0 : no seasonal influence  →  no seasonal component
  H1 : seasonal influence     →  presence of seasonal component S_t

Decision rule: reject H0 if F_c > F_{ν1;ν2} (α = 0.05)
               i.e. reject H0 if F_c > 1.8303

Results:
  P = 22 periods,  S = 12 seasons
  ν1 = S - 1 = 11
  ν2 = (P - 1)(S - 1) = 231
  V_s = 7372.4242
  V_R = 30.0354
  F_c = V_s / V_R = 245.4582
  F critical (ν1=11, ν2=231, α=0.05) = 1.8303

Conclusion:
  F_c = 245.4582 > F critical = 1.8303
  → H0 is rejected
  → Presence of a seasonal component S_t

Seasonality present: True


## Decomposition order — variance impact

**Constraint:** decomposition is done in the order of the components that have the **largest variance impact**. In most cases (≈ 90–95%), the trend has the largest variance.

### Approach

From the variance analysis of the Buys-Ballot table, compare the sums of squares:

| Component | Sum of squares |
|-----------|----------------|
| Trend $T_t$ (period / row effect) | $S_p = S \sum_p (X_{p,\circ} - X_{\circ\circ})^2$ |
| Seasonality $S_t$ (season / column effect) | $S_s = P \sum_s (X_{\circ,s} - X_{\circ\circ})^2$ |
| Residual $R_t$ | $S_R = \sum_p \sum_s (X_{p,s} - X_{p,\circ} - X_{\circ,s} + X_{\circ\circ})^2$ |

Share of total variance $S_T = S_s + S_p + S_R$:

$$\%T = \frac{S_p}{S_T},\quad \%S = \frac{S_s}{S_T},\quad \%R = \frac{S_R}{S_T}$$

**Rule:** extract components from the **highest** share to the **lowest** (residual last). Only components detected as present are included in the extraction order.

In [8]:
# Variance impact → decomposition order
bb_ord = df.copy()
bb_ord["year"] = bb_ord.index.year
bb_ord["month"] = bb_ord.index.month

counts = bb_ord.groupby("year").size()
complete = counts[counts == 12].index
bb_ord = bb_ord[bb_ord["year"].isin(complete)]

table_ord = bb_ord.pivot_table(index="year", columns="month", values="Interest")
X = table_ord.values
P, S = X.shape

X_p = X.mean(axis=1)
X_s = X.mean(axis=0)
X_oo = X.mean()

S_p = S * np.sum((X_p - X_oo) ** 2)  # trend (periodic)
S_s = P * np.sum((X_s - X_oo) ** 2)  # seasonality
S_R = np.sum((X - X_p[:, None] - X_s[None, :] + X_oo) ** 2)  # residual
S_T = S_s + S_p + S_R

pct_T = 100 * S_p / S_T
pct_S = 100 * S_s / S_T
pct_R = 100 * S_R / S_T

print("=" * 60)
print("Variance impact by component")
print("=" * 60)
print(f"  S_p (Trend T_t)        = {S_p:.4f}   ({pct_T:.2f}%)")
print(f"  S_s (Seasonality S_t)  = {S_s:.4f}   ({pct_S:.2f}%)")
print(f"  S_R (Residual R_t)     = {S_R:.4f}   ({pct_R:.2f}%)")
print(f"  S_T (Total)            = {S_T:.4f}   (100%)")

# Rank all components by variance share
impact = {
    "T_t (trend)": pct_T,
    "S_t (seasonality)": pct_S,
    "R_t (residual)": pct_R,
}
ranked = sorted(impact.items(), key=lambda x: x[1], reverse=True)

print("\nRanking (highest variance impact → lowest):")
for i, (name, pct) in enumerate(ranked, 1):
    print(f"  {i}. {name} — {pct:.2f}%")

# Extraction order: present components only, residual always last
# (uses trend_present / seasonality_present from the Fisher tests above)
extraction = []
candidates = []
if trend_present:
    candidates.append(("T_t", pct_T))
if seasonality_present:
    candidates.append(("S_t", pct_S))
candidates.sort(key=lambda x: x[1], reverse=True)
extraction = [c[0] for c in candidates] + ["R_t"]

decomposition_order = " → ".join(extraction)

print("\n" + "=" * 60)
print("Decomposition order (extract in this sequence)")
print("=" * 60)
print(f"  Detected: trend={trend_present}, seasonality={seasonality_present}")
print(f"  Order: {decomposition_order}")
print("=" * 60)

Variance impact by component
  S_p (Trend T_t)        = 13007.6515   (12.87%)
  S_s (Seasonality S_t)  = 81096.6667   (80.26%)
  S_R (Residual R_t)     = 6938.1667   (6.87%)
  S_T (Total)            = 101042.4848   (100%)

Ranking (highest variance impact → lowest):
  1. S_t (seasonality) — 80.26%
  2. T_t (trend) — 12.87%
  3. R_t (residual) — 6.87%

Decomposition order (extract in this sequence)
  Detected: trend=True, seasonality=True
  Order: S_t → T_t → R_t
